In [1]:
import os
import shutil
import pandas as pd
import rioxarray
import xarray as xr
from rasterio.enums import Resampling
import numpy as np
from scipy.ndimage import gaussian_filter
import sys
import time
import pickle
import numpy as np
import rasterio
import io
import geopandas as gpd
import contextlib
import function_process as process 
import onstove

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_DOWNLOAD_DIR = "OnStove_inputs_and_outputs"
GIS_DATA_DIR = os.path.join(BASE_DOWNLOAD_DIR, "GIS data")
INPUTS_DIR = os.path.join(BASE_DOWNLOAD_DIR, "Inputs")
TECH_SPECS_DIR = os.path.join(INPUTS_DIR, "Techno-economic files", "Private benefits")
SOCIO_SPECS_DIR = os.path.join(INPUTS_DIR, "Socio-economic files", "Private benefits")

OUTPUT_ROOT = "SSA"
TARGET_DIR = os.path.join(OUTPUT_ROOT, "all_inputSSA")
os.makedirs(TARGET_DIR, exist_ok=True)
# check that the threshold is the same for all the countries
URBAN_THRESHOLD = 20 

# Map 3-letter ISO to the safe names expected by your notebook
ISO_TO_COUNTRY = {
    'AGO': 'angola', 'BDI': 'burundi', 'BEN': 'benin', 'BFA': 'burkina_faso',
    'BWA': 'botswana', 'CAF': 'central_african_republic', 'CIV': 'ivory_coast',
    'CMR': 'cameroon', 'COD': 'democratic_republic_of_the_congo', 'COG': 'republic_of_the_congo',
    'DJI': 'djibouti', 'ERI': 'eritrea', 'ETH': 'ethiopia', 'GAB': 'gabon',
    'GHA': 'ghana', 'GIN': 'guinea', 'GMB': 'gambia', 'GNB': 'guinea-bissau',
    'GNQ': 'equatorial_guinea', 'KEN': 'kenya', 'LBR': 'liberia', 'LSO': 'lesotho',
    'MDG': 'madagascar', 'MLI': 'mali', 'MOZ': 'mozambique', 'MRT': 'mauritania',
    'MWI': 'malawi', 'NAM': 'namibia', 'NER': 'niger', 'NGA': 'nigeria',
    'RWA': 'rwanda', 'SDN': 'sudan', 'SEN': 'senegal', 'SLE': 'sierra_leone',
    'SOM': 'somalia', 'SSD': 'south_sudan', 'SWZ': 'eswatini', 'TCD': 'chad',
    'TGO': 'togo', 'TZA': 'united_republic_of_tanzania', 'UGA': 'uganda',
    'ZAF': 'south_africa', 'ZMB': 'zambia', 'ZWE': 'zimbabwe'
}
# relative wealth index is downlaoded from data.humdata.org ; missing south sudan, somalia, sudan.
# Add your specific GINI and GDP PC data here for the OnStove AWE calculation   gini from  2026 World Population Review; gdp per capita from: 2024 World Bank data 
COUNTRY_ECON_DATA = {
    'AGO': {'gini': 0.513, 'gdp_pc': 2665.9, 'pareto_weight': 0.32},  # angola
    'BDI': {'gini': 0.375, 'gdp_pc': 219.4, 'pareto_weight': 0.32},   # burundi
    'BEN': {'gini': 0.344, 'gdp_pc': 1485.4, 'pareto_weight': 0.32},  # benin
    'BFA': {'gini': 0.374, 'gdp_pc': 982.0, 'pareto_weight': 0.32},   # burkina_faso
    'BWA': {'gini': 0.549, 'gdp_pc': 7695.8, 'pareto_weight': 0.32},  # botswana
    'CAF': {'gini': 0.43,  'gdp_pc': 516.2, 'pareto_weight': 0.32},   # central_african_republic
    'CIV': {'gini': 0.353, 'gdp_pc': 2727.9, 'pareto_weight': 0.32},  # ivory_coast
    'CMR': {'gini': 0.422, 'gdp_pc': 1830.0, 'pareto_weight': 0.32},  # cameroon
    'COD': {'gini': 0.447, 'gdp_pc': 649.4, 'pareto_weight': 0.32},   # democratic_republic_of_the_congo
    'COG': {'gini': 0.489, 'gdp_pc': 2482.2, 'pareto_weight': 0.32},  # republic_of_the_congo
    'DJI': {'gini': 0.416, 'gdp_pc': 3552.7, 'pareto_weight': 0.32},  # djibouti
    'ERI': {'gini': 0.376, 'gdp_pc': 688.7, 'pareto_weight': 0.32},   # eritrea #gini value: confidus solutions, no year specified, assumed as proxy
    'ETH': {'gini': 0.311, 'gdp_pc': 1133.9, 'pareto_weight': 0.32},  # ethiopia
    'GAB': {'gini': 0.38,  'gdp_pc': 8230.0, 'pareto_weight': 0.32},  # gabon
    'GHA': {'gini': 0.435, 'gdp_pc': 2390.8, 'pareto_weight': 0.32},  # ghana
    'GIN': {'gini': 0.296, 'gdp_pc': 1695.0, 'pareto_weight': 0.32},  # guinea
    'GMB': {'gini': 0.388, 'gdp_pc': 871.3, 'pareto_weight': 0.32},   # gambia
    'GNB': {'gini': 0.334, 'gdp_pc': 1007.7, 'pareto_weight': 0.32},  # guinea-bissau
    'GNQ': {'gini': 0.385, 'gdp_pc': 6745.4, 'pareto_weight': 0.32},  # equatorial_guinea
    'KEN': {'gini': 0.377, 'gdp_pc': 2132.4, 'pareto_weight': 0.32},  # kenya
    'LBR': {'gini': 0.353, 'gdp_pc': 851.5, 'pareto_weight': 0.32},   # liberia
    'LSO': {'gini': 0.449, 'gdp_pc': 971.9, 'pareto_weight': 0.32},   # lesotho
    'MDG': {'gini': 0.425, 'gdp_pc': 545.0, 'pareto_weight': 0.32},   # madagascar
    'MLI': {'gini': 0.357, 'gdp_pc': 1094.6, 'pareto_weight': 0.32},  # mali
    'MOZ': {'gini': 0.496, 'gdp_pc': 656.8, 'pareto_weight': 0.32},   # mozambique
    'MRT': {'gini': 0.32,  'gdp_pc': 2110.1, 'pareto_weight': 0.32},  # mauritania
    'MWI': {'gini': 0.385, 'gdp_pc': 522.6, 'pareto_weight': 0.32},   # malawi
    'NAM': {'gini': 0.591, 'gdp_pc': 4413.1, 'pareto_weight': 0.32},  # namibia
    'NER': {'gini': 0.329, 'gdp_pc': 735.3, 'pareto_weight': 0.32},   # niger
    'NGA': {'gini': 0.339, 'gdp_pc': 2139.0, 'pareto_weight': 0.32},  # nigeria  #value 2022 of gdp before svalutation of naira
    'RWA': {'gini': 0.394, 'gdp_pc': 999.7, 'pareto_weight': 0.32},   # rwanda
    'SDN': {'gini': 0.342, 'gdp_pc': 984.6, 'pareto_weight': 0.32},   # sudan
    'SEN': {'gini': 0.362, 'gdp_pc': 1773.2, 'pareto_weight': 0.32},  # senegal
    'SLE': {'gini': 0.357, 'gdp_pc': 806.7, 'pareto_weight': 0.32},   # sierra_leone
    'SOM': {'gini': 0.352, 'gdp_pc': 629.5, 'pareto_weight': 0.32},   # somalia    #gini value: Somalia National Bureau of Statistics, value from year 2022
    'SSD': {'gini': 0.44,  'gdp_pc': 1080.1, 'pareto_weight': 0.32},  # south_sudan
    'SWZ': {'gini': 0.546, 'gdp_pc': 3909.6, 'pareto_weight': 0.32},  # eswatini
    'TCD': {'gini': 0.374, 'gdp_pc': 961.6, 'pareto_weight': 0.32},   # chad
    'TGO': {'gini': 0.379, 'gdp_pc': 1119.4, 'pareto_weight': 0.32},  # togo
    'TZA': {'gini': 0.405, 'gdp_pc': 1186.7, 'pareto_weight': 0.32},  # tanzania
    'UGA': {'gini': 0.427, 'gdp_pc': 1077.9, 'pareto_weight': 0.32},  # uganda
    'ZAF': {'gini': 0.63,  'gdp_pc': 6267.2, 'pareto_weight': 0.32},  # south_africa
    'ZMB': {'gini': 0.515, 'gdp_pc': 1187.1, 'pareto_weight': 0.32},  # zambia
    'ZWE': {'gini': 0.503, 'gdp_pc': 2497.2, 'pareto_weight': 0.32},  # zimbabwe
}

# ── Helper: Safe String ───────────────────────────────────────────────────────
def safe_name(country_name):
    return (country_name.lower()
            .replace(' ', '_').replace('.', '').replace("'", '')
            .replace('é', 'e').replace('ô', 'o').replace('ã', 'a')
            .replace('ç', 'c').replace('ó', 'o'))


# ── Define Custom Income Raster Generator ─────────────────────────────────────
def custom_generate_income_raster(model_pickle_path, scenario_csv_path, output_directory, output_raster_name, gini, gdp_pc, pareto_weight):
    """Estimates income utilizing the OnStove AWE method and outputs a base income raster."""
    os.makedirs(output_directory, exist_ok=True)
    
    # Load the specific country's calibrated model
    model = onstove.OnStove.read_model(model_pickle_path)
    model.read_scenario_data(scenario_csv_path, delimiter=",")
    model.output_directory = output_directory

    # Inject the country-specific economic data
    model.specs["gini"] = float(gini)
    model.specs["gdp_pc"] = float(gdp_pc)

    with contextlib.redirect_stdout(io.StringIO()):
        model.income_estimation(awe=True, income_data=None, pareto_weight=pareto_weight)
        model.to_raster(variable="absolute_wealth")

    src_name = "absolute_wealth_mean.tif"
    src_path = os.path.join(output_directory, "Rasters", src_name)
    dst_path = os.path.join(output_directory, f"{output_raster_name}.tif")

    if os.path.exists(src_path):
        if os.path.exists(dst_path):
            os.remove(dst_path)
        os.replace(src_path, dst_path)
        
    rasters_dir = os.path.join(output_directory, "Rasters")
    if os.path.exists(rasters_dir):
        shutil.rmtree(rasters_dir, ignore_errors=True)


# ── Main Extraction Loop ──────────────────────────────────────────────────────
extracted_lpg_shares = {}

for iso, country_full in ISO_TO_COUNTRY.items():
    print(f"\nProcessing {iso} -> {country_full}")
    s_name = safe_name(country_full)
    iso_gis_dir = os.path.join(GIS_DATA_DIR, iso)
    # --- FORCE REDO EXCEPTIONS LOGIC ---
    final_income_path = os.path.join(TARGET_DIR, f"income_{s_name}.tif")
    
    # If the final income raster exists, the country is 100% complete.
    if os.path.exists(final_income_path):
        print(f"[SKIP] {iso} -> Already perfectly completed. Fetching specs and skipping.")
        
        # --- ADDED: Extract LPG Shares for Skipped Countries ---
        tech_csv = os.path.join(TECH_SPECS_DIR, f"{iso}_file_tech_specs.csv")
        urban_share, rural_share = 0.4321, 0.4321  # Default fallbacks
        if os.path.exists(tech_csv):
            df_tech = pd.read_csv(tech_csv, header=None)
            lpg_rows = df_tech[df_tech[0] == 'LPG']
            try:
                rural_share = float(lpg_rows[lpg_rows[1] == 'current_share_rural'][2].values[0])
                urban_share = float(lpg_rows[lpg_rows[1] == 'current_share_urban'][2].values[0])
            except IndexError:
                pass
        
        extracted_lpg_shares[s_name] = {
            'urban_lpg_share': urban_share,
            'rural_lpg_share': rural_share
        }
        # --------------------------------------------------------
        continue
        
    # If the raster does NOT exist, force the code to redo the ENTIRE pipeline.
    print(f"[REDO TARGET] {iso} -> Missing final income raster. Re-running entire pipeline properly...")

    # 1. Copy Population Raster
    pop_src = os.path.join(iso_gis_dir, "Demographics", "Population", "Population.tif")
    pop_dst = os.path.join(TARGET_DIR, f"population_{s_name}.tif")
    if os.path.exists(pop_src):
        shutil.copy2(pop_src, pop_dst)
    else:
        print(f"  ⚠ Missing Population for {iso}")

    # 2. Copy Urban Raster
    urban_src = os.path.join(iso_gis_dir, "Demographics", "Urban", "Urban.tif")
    urban_dst = os.path.join(TARGET_DIR, f"urban_{s_name}.tif")
    if os.path.exists(urban_src):
        shutil.copy2(urban_src, urban_dst)
    else:
        print(f"  ⚠ Missing Urban for {iso}")

    # 3. Copy Friction Raster
    friction_dir = os.path.join(iso_gis_dir, "Biomass", "Friction")
    friction_dst = os.path.join(TARGET_DIR, f"friction_{s_name}.tif")
    if os.path.exists(friction_dir):
        f_files = [f for f in os.listdir(friction_dir) if f.endswith('.tif')]
        if f_files:
            shutil.copy2(os.path.join(friction_dir, f_files[0]), friction_dst)
        else:
            print(f"  ⚠ Missing Friction tif for {iso}")
    else:
        print(f"  ⚠ Missing Friction directory for {iso}")

    # 4. Extract LPG Shares from tech_specs.csv
    tech_csv = os.path.join(TECH_SPECS_DIR, f"{iso}_file_tech_specs.csv")
    urban_share, rural_share = 0.4321, 0.4321  # Default fallbacks
    
    if os.path.exists(tech_csv):
        df_tech = pd.read_csv(tech_csv, header=None)
        lpg_rows = df_tech[df_tech[0] == 'LPG']
        try:
            rural_share = float(lpg_rows[lpg_rows[1] == 'current_share_rural'][2].values[0])
            urban_share = float(lpg_rows[lpg_rows[1] == 'current_share_urban'][2].values[0])
        except IndexError:
            print(f"  ⚠ Could not parse LPG shares for {iso}, using defaults.")
    
    extracted_lpg_shares[s_name] = {
        'urban_lpg_share': urban_share,
        'rural_lpg_share': rural_share
    }
    
    # 5. Preparation of file for income generation, running onstove calibration
    output_directory = os.path.join(TARGET_DIR, f"{iso}_PreCalib_Workspace")
    os.makedirs(output_directory, exist_ok=True)
    
    # Step 1: Data Processing
    print("-> Step 1: Processing geographic datasets...")
    data = onstove.DataProcessor(project_crs=3395, cell_size=(1000, 1000))
    data.output_directory = output_directory
    
    # Dynamic path for the specific country's boundary file
    adm_path = os.path.join(GIS_DATA_DIR, f"{iso}/Administrative/Country_boundaries/Country_boundaries.geojson")
    
    if os.path.exists(adm_path):
        boundary = gpd.read_file(adm_path)
        # Ensure CRS matches
        if boundary.crs != "EPSG:3395":
            boundary = boundary.to_crs("EPSG:3395")
        
        # Unify into a single polygon to satisfy DataProcessor
        mask_boundary = boundary.dissolve()
        temp_mask_path = os.path.join(output_directory, f"mask_{iso}.geojson")
        mask_boundary.to_file(temp_mask_path, driver='GeoJSON')
        
        data.add_mask_layer(
            category='Administrative', 
            name='Country_boundaries', 
            path=temp_mask_path
        )
    else:
        print(f"  ⚠️ Warning: Boundary file not found at {adm_path}")
        
    
    pop_path = os.path.join(GIS_DATA_DIR, f"{iso}/Demographics/Population/Population.tif")
    urban_path = os.path.join(GIS_DATA_DIR, f"{iso}/Demographics/Urban/Urban.tif")
    mv_path = os.path.join(GIS_DATA_DIR, f"{iso}/Electricity/MV_lines/MV_lines.geojson")
    ntl_path = os.path.join(GIS_DATA_DIR, f"{iso}/Electricity/Night_time_lights/Night_time_lights.tif")
    lpg_path = os.path.join(GIS_DATA_DIR, f"{iso}/LPG/Traveltime/Traveltime.tif")
    
    data.add_layer(category='Demographics', name='Population', path=pop_path, layer_type='raster', resample='sum')
    data.add_layer(category='Demographics', name='Urban_rural_divide', path=urban_path, layer_type='raster', resample='nearest')
    data.add_layer(category='Electricity', name='MV_lines', path=mv_path, layer_type='vector')
    data.add_layer(category='Electricity', name='Night_time_lights', path=ntl_path, layer_type='raster', resample='average')
    data.add_layer(category='LPG', name='LPG Traveltime', path=lpg_path, layer_type='raster', resample='average')
    
    data.add_layer(category='Base', name='Base', path=pop_path, layer_type='raster', base_layer=True, resample='nearest')
    
    data.align_layers(datasets='all')
    data.reproject_layers(datasets={'Electricity': ['MV_lines']})
    data.save_datasets('all')
    
    # Step 2: OnStove Model Creation & Calibration Initialization
    print("-> Step 2: Calibrating OnStove framework specifications...")
    country_model = onstove.OnStove(project_crs=3395)
    country_model.output_directory = output_directory
    
    soc_path = os.path.join(SOCIO_SPECS_DIR, f"{iso}.csv") 
    tech_path = os.path.join(TECH_SPECS_DIR, f"{iso}_file_tech_specs.csv")
    
    if not os.path.exists(soc_path) or not os.path.exists(tech_path):
        print(f"  ⚠️ Skipping {iso}: Configuration spec sheets matching files were not found.")
        continue
        
    country_model.read_scenario_data(soc_path, delimiter=',')
    
    calib_pop_path = os.path.join(output_directory, 'Demographics', 'Population', 'Population.tif')
    country_model.add_layer(category='Demographics', name='Population', path=calib_pop_path, layer_type='raster', base_layer=True)
    country_model.population_to_dataframe()
    
    ghs_path = os.path.join(output_directory, 'Demographics', 'Urban_rural_divide', 'Urban_rural_divide.tif')
    country_model.calibrate_urban_rural_split(ghs_path)
    
    wealth_index = os.path.join(GIS_DATA_DIR, f"All_wealth_index/{iso}_relative_wealth_index.csv") 
    if os.path.exists(wealth_index):
        # Read as a regular dataframe
        df_rwi = pd.read_csv(wealth_index)  #skipping onstove's built in method since it expects a quadkey
        
        # Convert to GeoDataFrame
        gdf_rwi = gpd.GeoDataFrame(
            df_rwi, 
            geometry=gpd.points_from_xy(df_rwi.longitude, df_rwi.latitude),
            crs="EPSG:4326"
        )
        # Reproject to your model's CRS
        gdf_rwi = gdf_rwi.to_crs(country_model.gdf.crs)
        
        # Map to model grid using spatial join
        country_model.gdf = gpd.sjoin_nearest(country_model.gdf, gdf_rwi, how='left')
        
        # Rename the column so OnStove recognizes it
        if 'rwi' in country_model.gdf.columns:
            country_model.gdf.rename(columns={'rwi': 'relative_wealth'}, inplace=True)
    else:
        print(f" ⚠️ Warning: RWI file missing for {iso}.")
    
    mv_geojson_path = os.path.join(output_directory, 'Electricity', 'MV_lines', 'MV_lines.geojson')
    mv_lines = onstove.VectorLayer('Electricity', 'MV_lines', path=mv_geojson_path)
    country_model.distance_to_electricity(mv_lines=mv_lines)
    
    ntl_calib_path = os.path.join(output_directory, 'Electricity', 'Night_time_lights', 'Night_time_lights.tif')
    country_model.raster_to_dataframe(ntl_calib_path, name='Night_lights', method='read')
    
    country_model.current_elec()
    
    country_model.read_tech_data(tech_path, delimiter=',')
    country_model.techs['Electricity'].get_capacity_cost(country_model)
    
    traveltime_path = os.path.join(output_directory, 'LPG', 'LPG Traveltime', 'LPG Traveltime.tif')
    country_model.techs['LPG'].travel_time = country_model.raster_to_dataframe(
        traveltime_path, fill_nodata_method='interpolate', method='read'
    ) * 2 / 60
    
    print(f"-> Step 3: Base calibration ready. Freezing 'model.input.pkl' for {iso}.")
    pkl_filename = os.path.join(output_directory, f"{iso}_model.input.pkl")
    with open(pkl_filename, 'wb') as f:
        pickle.dump(country_model, f)
        
    print(f"  Finished setup phase for income for {country_full}. Input file saved to {pkl_filename}")

    # 6. Income Generation (Now executing AFTER the pkl is successfully saved)
    econ_data = COUNTRY_ECON_DATA.get(iso)
    
    if econ_data and econ_data['gdp_pc'] is not None and econ_data['gini'] is not None:
        # ── SAFETY CHECK: ONLY RUN IF WEALTH INDEX WAS FOUND ──────────────────
        if 'relative_wealth' in country_model.gdf.columns:
            print(f"  Generating Income Raster for {iso}...")
            raw_income_name = "income_raw"
                        
            # Call the standalone generator function natively defined in this notebook
            custom_generate_income_raster(
                model_pickle_path=pkl_filename, 
                scenario_csv_path=soc_path,
                output_directory=TARGET_DIR,
                output_raster_name=raw_income_name,
                gini=econ_data['gini'],
                gdp_pc=econ_data['gdp_pc'],
                pareto_weight=econ_data.get('pareto_weight', 0.32)
            )
            
            raw_income_path = os.path.join(TARGET_DIR, f"{raw_income_name}.tif")
            if os.path.exists(raw_income_path) and os.path.exists(urban_dst) and os.path.exists(pop_dst):
                print(f"  Running AWE Limitation Recovery...")
                
                income_da = rioxarray.open_rasterio(raw_income_path)
                
                # Use process.py for the recovery step as requested
                data_imputed, valid_mask = process.awe_limitation_recovery(
                    income_da=income_da,
                    urban_raster_path=urban_dst,
                    output_nodata=np.nan,
                    population_raster_path=pop_dst
                )
                
                final_income_da = income_da.copy(data=data_imputed)
                final_income_path = os.path.join(TARGET_DIR, f"income_{s_name}.tif")
                final_income_da.rio.to_raster(final_income_path)
                income_da.close()
                
                os.remove(raw_income_path)
        else:
            print(f"  Skipping ONLY Income Raster for {iso}: 'relative_wealth' data was missing")
    else:
        print(f"  Skipping Income Generation: Missing valid econ data (GDP or GINI) for {iso}")


# ── Output Extracted Shares for the Notebook ──────────────────────────────────
print("\n\nExtraction Complete. Paste the following dictionary into your notebook's SSA_COUNTRIES block:")
print("SSA_COUNTRIES = {")
for country, shares in extracted_lpg_shares.items():
    print(f"    '{country}': {{'urban_lpg_share': {shares['urban_lpg_share']:.4f}, 'rural_lpg_share': {shares['rural_lpg_share']:.4f}}},")
print("}")


Processing AGO -> angola
[SKIP] AGO -> Already perfectly completed. Fetching specs and skipping.

Processing BDI -> burundi
[SKIP] BDI -> Already perfectly completed. Fetching specs and skipping.

Processing BEN -> benin
[SKIP] BEN -> Already perfectly completed. Fetching specs and skipping.

Processing BFA -> burkina_faso
[SKIP] BFA -> Already perfectly completed. Fetching specs and skipping.

Processing BWA -> botswana
[SKIP] BWA -> Already perfectly completed. Fetching specs and skipping.

Processing CAF -> central_african_republic
[SKIP] CAF -> Already perfectly completed. Fetching specs and skipping.

Processing CIV -> ivory_coast
[SKIP] CIV -> Already perfectly completed. Fetching specs and skipping.

Processing CMR -> cameroon
[SKIP] CMR -> Already perfectly completed. Fetching specs and skipping.

Processing COD -> democratic_republic_of_the_congo
[SKIP] COD -> Already perfectly completed. Fetching specs and skipping.

Processing COG -> republic_of_the_congo
[SKIP] COG -> Alre